# vLLM Prefix Cache Comparison

This notebook compares the same CoT inference flow from `homework/vllm_cot.py` with vLLM Automatic Prefix Caching disabled and enabled.

The workload is a good fit for prefix caching because every question uses the same system prompt and few-shot CoT examples; only the final user question changes. vLLM's Automatic Prefix Caching reuses the KV cache for a shared prompt prefix when `enable_prefix_caching=True`.

Reference: https://docs.vllm.ai/en/latest/features/automatic_prefix_caching/

Run this in a Linux/CUDA environment with vLLM installed. It is not expected to run on Mac MPS.

In [ ]:
# Optional install cell. Run only in an environment where vLLM is supported.
# %pip install vllm pandas matplotlib

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys
import tempfile

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None


def find_file(relative_paths):
    for relative_path in relative_paths:
        candidate = Path.cwd() / relative_path
        if candidate.exists():
            return candidate.resolve()
    tried = ", ".join(str(Path.cwd() / path) for path in relative_paths)
    raise FileNotFoundError(f"Could not find any of: {tried}")


module_path = find_file([
    Path("homework") / "vllm_cot.py",
    Path("CoT") / "homework" / "vllm_cot.py",
])
data_path = find_file([
    Path("data") / "valid.json",
    Path("CoT") / "data" / "valid.json",
])

spec = importlib.util.spec_from_file_location("vllm_cot_module", module_path)
vc = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(vc)

module_path, data_path

## Environment Check

The benchmark child process uses the same Python executable as this notebook kernel. If this cell cannot import `vllm`, switch to a Linux/CUDA kernel where vLLM is installed before running the benchmark.

In [ ]:
vllm_spec = importlib.util.find_spec("vllm")
env_info = {
    "python_executable": sys.executable,
    "vllm_installed": vllm_spec is not None,
}

if vllm_spec is not None:
    import vllm
    env_info["vllm_version"] = getattr(vllm, "__version__", "unknown")

env_info

if vllm_spec is None:
    raise ModuleNotFoundError(
        "vLLM is not installed in this notebook kernel. "
        "Install it in a Linux/CUDA environment, then select that environment as the Jupyter kernel. "
        "For example: python -m pip install vllm pandas matplotlib"
    )

## Benchmark Settings

`warmup_questions` uses different questions with the same shared CoT prefix. That lets the prefix-cache run populate its cache before the timed section without timing model load or first-use setup.

In [ ]:
checkpoint = vc.DEFAULT_CHECKPOINT
num_questions = 96
warmup_questions = 8
batch_size = 8
max_tokens = 75
temperature = 0.0
repetitions = 1

tensor_parallel_size = 1
dtype = "auto"
gpu_memory_utilization = 0.9
max_model_len = None
max_num_seqs = None
trust_remote_code = True

In [ ]:
BENCHMARK_PROGRAM = r'''
import gc
import importlib.util
import inspect
import json
import math
import os
import sys
import time
from pathlib import Path

module_path = Path(os.environ["MODULE_PATH"])
data_path = Path(os.environ["DATA_PATH"])
enable_prefix_caching = os.environ["ENABLE_PREFIX_CACHING"] == "1"

checkpoint = os.environ["CHECKPOINT"]
num_questions = int(os.environ["NUM_QUESTIONS"])
warmup_questions = int(os.environ["WARMUP_QUESTIONS"])
batch_size = int(os.environ["BATCH_SIZE"])
max_tokens = int(os.environ["MAX_TOKENS"])
temperature = float(os.environ["TEMPERATURE"])
tensor_parallel_size = int(os.environ["TENSOR_PARALLEL_SIZE"])
dtype = os.environ["DTYPE"]
gpu_memory_utilization = float(os.environ["GPU_MEMORY_UTILIZATION"])
max_model_len = os.environ["MAX_MODEL_LEN"]
max_num_seqs = os.environ["MAX_NUM_SEQS"]
trust_remote_code = os.environ["TRUST_REMOTE_CODE"] == "1"

spec = importlib.util.spec_from_file_location("vllm_cot_module", module_path)
vc = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(vc)

with data_path.open() as f:
    rows = json.load(f)

questions = [row[0] for row in rows]
warmup = questions[:warmup_questions]
timed_questions = questions[warmup_questions:warmup_questions + num_questions]

from vllm import LLM
try:
    import vllm
    vllm_version = getattr(vllm, "__version__", "unknown")
except Exception:
    vllm_version = "unknown"

llm_signature = inspect.signature(LLM.__init__)
llm_parameters = llm_signature.parameters
llm_accepts_kwargs = any(param.kind == inspect.Parameter.VAR_KEYWORD for param in llm_parameters.values())
prefix_cache_arg_supported = "enable_prefix_caching" in llm_parameters or llm_accepts_kwargs
print("DIAGNOSTIC_JSON " + json.dumps({
    "python": sys.version.split()[0],
    "vllm_version": vllm_version,
    "enable_prefix_caching_requested": enable_prefix_caching,
    "enable_prefix_caching_supported": prefix_cache_arg_supported,
    "llm_init_signature": str(llm_signature),
}))
if not prefix_cache_arg_supported:
    raise TypeError(
        "This vLLM installation does not expose enable_prefix_caching in LLM.__init__. "
        "Upgrade vLLM or use a version that supports Automatic Prefix Caching."
    )


engine_kwargs = {
    "model": checkpoint,
    "tensor_parallel_size": tensor_parallel_size,
    "dtype": dtype,
    "gpu_memory_utilization": gpu_memory_utilization,
    "trust_remote_code": trust_remote_code,
    "enable_prefix_caching": enable_prefix_caching,
}
if max_model_len != "None":
    engine_kwargs["max_model_len"] = int(max_model_len)
if max_num_seqs != "None":
    engine_kwargs["max_num_seqs"] = int(max_num_seqs)

model = vc.VLLMCoTModel.__new__(vc.VLLMCoTModel)
model.llm = LLM(**engine_kwargs)
model.tokenizer = model.llm.get_tokenizer()

def chunks(items, size):
    for start in range(0, len(items), size):
        yield items[start:start + size]

def common_prefix_chars(left, right):
    limit = min(len(left), len(right))
    for index in range(limit):
        if left[index] != right[index]:
            return index
    return limit

sample_prompt = model.format_prompt(timed_questions[0])
second_prompt = model.format_prompt(timed_questions[1]) if len(timed_questions) > 1 else sample_prompt
shared_chars = common_prefix_chars(sample_prompt, second_prompt)
shared_tokens = len(model.tokenizer.encode(sample_prompt[:shared_chars]))
prompt_tokens = len(model.tokenizer.encode(sample_prompt))

if warmup:
    for batch in chunks(warmup, batch_size):
        model.batched_generate(batch, temperature=temperature, max_tokens=max_tokens)

generations = []
start_time = time.perf_counter()
for batch in chunks(timed_questions, batch_size):
    generations.extend(model.batched_generate(batch, temperature=temperature, max_tokens=max_tokens))
elapsed_s = time.perf_counter() - start_time

answers = [model.parse_answer(generation) for generation in generations]
answer_rate = sum(math.isfinite(answer) for answer in answers) / len(answers)

result = {
    "enable_prefix_caching": enable_prefix_caching,
    "num_questions": len(timed_questions),
    "batch_size": batch_size,
    "elapsed_s": elapsed_s,
    "qps": len(timed_questions) / elapsed_s,
    "s_per_question": elapsed_s / len(timed_questions),
    "answer_rate": answer_rate,
    "prompt_tokens": prompt_tokens,
    "shared_prefix_tokens": shared_tokens,
    "sample_generation": generations[0] if generations else "",
    "vllm_version": vllm_version,
    "prefix_cache_arg_supported": prefix_cache_arg_supported,
}

try:
    del model.llm
except Exception:
    pass
gc.collect()

print("RESULT_JSON " + json.dumps(result, ensure_ascii=True))
'''


def run_benchmark_child(enable_prefix_caching: bool):
    env = os.environ.copy()
    env.update({
        "MODULE_PATH": str(module_path),
        "DATA_PATH": str(data_path),
        "ENABLE_PREFIX_CACHING": "1" if enable_prefix_caching else "0",
        "CHECKPOINT": checkpoint,
        "NUM_QUESTIONS": str(num_questions),
        "WARMUP_QUESTIONS": str(warmup_questions),
        "BATCH_SIZE": str(batch_size),
        "MAX_TOKENS": str(max_tokens),
        "TEMPERATURE": str(temperature),
        "TENSOR_PARALLEL_SIZE": str(tensor_parallel_size),
        "DTYPE": dtype,
        "GPU_MEMORY_UTILIZATION": str(gpu_memory_utilization),
        "MAX_MODEL_LEN": str(max_model_len),
        "MAX_NUM_SEQS": str(max_num_seqs),
        "TRUST_REMOTE_CODE": "1" if trust_remote_code else "0",
    })
    env.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "fork")
    with tempfile.TemporaryDirectory(prefix="vllm_cache_bench_") as tmpdir:
        script_path = Path(tmpdir) / "benchmark_child.py"
        script_path.write_text(BENCHMARK_PROGRAM)
        proc = subprocess.run(
            [sys.executable, str(script_path)],
            env=env,
            text=True,
            capture_output=True,
            check=False,
        )
    if proc.returncode != 0:
        stdout_tail = proc.stdout[-6000:] if proc.stdout else "(empty)"
        stderr_tail = proc.stderr[-6000:] if proc.stderr else "(empty)"
        raise RuntimeError(
            f"Benchmark child failed with exit code {proc.returncode}\n\n"
            f"--- child stdout tail ---\n{stdout_tail}\n\n"
            f"--- child stderr tail ---\n{stderr_tail}"
        )

    for line in reversed(proc.stdout.splitlines()):
        if line.startswith("RESULT_JSON "):
            return json.loads(line.removeprefix("RESULT_JSON "))

    stdout_tail = proc.stdout[-6000:] if proc.stdout else "(empty)"
    stderr_tail = proc.stderr[-6000:] if proc.stderr else "(empty)"
    raise RuntimeError(
        "Benchmark child did not print RESULT_JSON\n\n"
        f"--- child stdout tail ---\n{stdout_tail}\n\n"
        f"--- child stderr tail ---\n{stderr_tail}"
    )

## Run Comparison

Each row below starts a fresh Python process and creates one vLLM engine with the requested cache setting. Model load and warmup are excluded from `elapsed_s`; only the timed generation loop is measured.

In [ ]:
results = []
for repetition in range(1, repetitions + 1):
    for enable_prefix_caching in (False, True):
        print(f"Running repetition={repetition}, enable_prefix_caching={enable_prefix_caching}...")
        row = run_benchmark_child(enable_prefix_caching)
        row["repetition"] = repetition
        results.append(row)

if pd is not None:
    df = pd.DataFrame(results)
    display(df.drop(columns=["sample_generation"]))
else:
    print(json.dumps([{k: v for k, v in row.items() if k != "sample_generation"} for row in results], indent=2))

In [ ]:
def mean(values):
    return sum(values) / len(values)

no_cache_elapsed = mean([row["elapsed_s"] for row in results if not row["enable_prefix_caching"]])
cache_elapsed = mean([row["elapsed_s"] for row in results if row["enable_prefix_caching"]])
speedup = no_cache_elapsed / cache_elapsed

summary = {
    "no_cache_elapsed_s": no_cache_elapsed,
    "cache_elapsed_s": cache_elapsed,
    "speedup_x": speedup,
    "shared_prefix_tokens": results[0]["shared_prefix_tokens"],
    "prompt_tokens": results[0]["prompt_tokens"],
}
summary

In [ ]:
if plt is not None:
    labels = ["No prefix cache", "Prefix cache"]
    elapsed_values = [no_cache_elapsed, cache_elapsed]
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(labels, elapsed_values, color=["#5b6472", "#2f9e44"])
    ax.set_ylabel("Timed generation seconds")
    ax.set_title(f"vLLM CoT prefix cache comparison ({speedup:.2f}x speedup)")
    ax.bar_label(bars, fmt="%.2f s")
    plt.show()
else:
    print("Install matplotlib to show a chart.")

## Reading The Result

Prefix caching should help most when the shared prompt prefix is long and many requests reuse it. In this CoT example, the shared prefix is the system prompt plus the few-shot examples. If the measured speedup is small, common reasons are: the prompt prefix is short relative to generated tokens, batch size is too large for the benchmark shape, the GPU is already decode-bound, or the vLLM/version/backend has different cache defaults.

For a more obvious cache effect, increase `num_questions`, lower `batch_size`, or make the shared few-shot prefix longer.